# SSW 2026 I Group Project: Single Point Lens Microlensing

Casey Lam (Carnegie), Ali Crisp (OSU), Scott Gaudi (OSU), Katarzyna Kruszynska (LCO), Amber Malpas (OSU),  Arjun Mulidhar (OSU), Himanshu Verma(LSU)

Prior to running this notebook, be sure to run the setup notebook:  SSW2026_HandsOnI_Microlens_Setup.ipynb. The setup notebook just needs to be run **once**.

## Google Colab Usage

**Confirm login account**
* Please make sure to be logged in with the Google account you want to use for the exercises before running the code cells below. You can check by clicking the circular account icon in the top right corner of the colab notebook.

**Google Drive directory**
* Note: data will be copied in a directory called "SSW2026/Microlens" in your Google drive by the Setup notebook.

**Running cells**
* Run cells individually by clicking on the triangle on each cell
* For text cells, you can click the down arrow to get to the next cell

**To Restart runtime**
*   Click on Runtime menu item
*   Select Restart session
*   Select Run code cells individually from the top

**To Recreate runtime**
*   Click on Runtime menu item
*   Select Disconnect and Delete runtime
*   Select Run code cells individually from the top

**To Exit:**
*   Close the browser window

# `Sagan 2026 Group Project:` Single Point Lens Microlensing
`Objective:`
1. **Load and visualize** a simulated Roman-like microlensing light curve.
2. **Rough estimate of initial parameters** using the light curve data.
3. Fit a **PSPL** (Point Source Point Lens) model (Minimizer). Obtain best fit model and report parameters with minimum $\chi^2$, degree of freedom (dof), and z-score.
4. Run **MCMC for PSPL**: Visualize chains, posterior corner plots, and fitted light curve. Report parameters and their respective errors with minimum $\chi^2$, degree of freedom (dof), and z-score.
5. Test a higher-order effect model — **Finite-Source, Parallax, or Both** (Minimizer). *Requires data reduction.*
6. **Compare** the higher-order effect models with PSPL using their minimum $\chi^2$, $\chi^2$/dof, and z-score.
7. OPTIONAL: Run MCMC for the higher-order model

**Complete the code in all sections containing `___` and follow the `# TODO:` hints.**

---
## Quick-review: Microlensing Parameters & MulensModel
Before we begin, let's review the core parameters of a microlensing event and typical values for a Galactic bulge event:

**PSPL model Parameters:**
* **$t_0$**: Time of closest approach (peak magnification).
* **$u_0$**: Impact parameter (minimum distance between lens and source, normalized by the Einstein angle $\theta_E$). *Typical range: ~0 (high mag) to 1.0 (low mag).*
* **$t_E$**: Einstein ring crossing time. *Typical range: ~a few days to 100+ days.*
* **$fs$**: The source flux fraction ($f_s$) is simply the ratio of the magnifiable light from the lensing star to the total observed baseline light ($f_s = F_s / [F_s + F_b]$), which might contain unrelated blending from other stars in the crowded field. *Typical range: 0-1.*

Note: The Einstein angle is a related to the lens mass ($M_L$) and the distances to both the lens ($D_L$) and the source($D_S$), expressed as $\theta_E = \sqrt{\frac{4 G M_L}{c^2}\left(\frac{1}{D_L} - \frac{1}{D_S}\right)}$. The Einstein crossing time as $t_E=\frac{\theta_E}{\mu_\textrm{rel}}$, where $\mu_\textrm{rel}$ is the relative proper motion between the lens and the source.

**Higher-Order model additional Parameters:**
* **$\rho$ (FSPL)**: Normalized source radius ($\theta_* / \theta_E$). Finite source effects manifest strongly when the lens transits the source ($u_0 \lesssim \rho$). *Typical range: $10^{-3}$ to $1$.*
* **$\pi_{E,N}, \pi_{E,E}$ (Parallax)**: North and East components of the microlens parallax vector, normalized by the Einstein angle. This accounts for the Earth's orbital motion deviating from a straight line during the event. *Typical range: -1 to 1*
* fixed parameters--**t_0_ref, ra, dec**: Reference time, R.A. and Dec. for parallax (remain fixed)


**MulensModel Core Modules:**
Throughout this notebook, you will use three main classes from the `MulensModel` framework:
1.  `mm.MulensData`: object holds discrete lightcurve data: time, magnitude or flux, and its uncertainty (in that order). Full documentation at https://rpoleski.github.io/MulensModel/MulensModel.mulensdata.html#module-MulensModel.mulensdata
2.  `mm.Model`: defines a microlensing model and stores the physical parameters and computes the theoretical magnification $A(t)$. Full documentation is at https://rpoleski.github.io/MulensModel/MulensModel.model.html#module-MulensModel.model
3.  `mm.Event`: object takes a `Model` object and a `MulensData` object and links them together. This is useful because it enables the calculation of the $\chi^2$ goodness of fit statistic.Full documentation is at https://rpoleski.github.io/MulensModel/MulensModel.event.html#module-MulensModel.event . We can instantiate an `Event` object using the `Model` and `MulensData` objects created.

In [ ]:
# Install software
%pip install --quiet MulensModel emcee corner iminuit

# Colab widget bug work-around
get_ipython().kernel.do_shutdown(restart=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.7 MB/s eta 0:00:00


{'status': 'ok', 'restart': True}

###❗ **Restart session** ❗
You may need to restart the session - this was likely already done - you may have seen a runtime has crashed message which was the auto restart. If not, you can do a manual restart  (dropdown menu next to "Run all" button) After you restart the session, don't re-run the above cells, just continue with the next cell.

In [ ]:
# COLAB-ONLY
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import MulensModel as mm
import pandas as pd
import emcee
import corner
from iminuit import minimize

In [ ]:
# You will be prompted to Permit this notebook to access your Google Drive files - Click on "Connect to Google Drive"
# You will then be prompted to Choose an account - click on your preferred Google account
# You will then confirm that Google Drive for desktop wants to access your Google Account - select all and scroll to click "Continue"
# You may get another prompt to allow additional access for this to work - scroll to click "Continue"

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Note - this directory was created by the setup notebook but defined in
# this notebook so the files can be found by this notebook

ssw_dir = 'SSW2026/Microlens' #@param {type:"string"}

In [ ]:
# Define directory
import os
import sys

# Google top level drive dir
drive_dir = "/content/drive/MyDrive/"

# SSW dir path
ssw_path = os.path.join(drive_dir, ssw_dir)

# pointlensdata path
pointlensdata_path = os.path.join(ssw_path, 'pointlensdata')

## Helper Functions
*(Provided for you - run this cell to initialize)*

In [ ]:
def chi2_fun(theta, parameters_to_fit, event):
    """
    Calculate chi2 for given values of parameters.

    Keywords :
        theta: *np.ndarray*
            Vector of parameter values, e.g.,
            `np.array([5380., 0.5, 20.])`.

        parameters_to_fit: *list* of *str*
            List of names of parameters corresponding to theta, e.g.,
            `['t_0', 'u_0', 't_E']`.

        event: *MulensModel.Event*
            Event which has datasets for which chi2 will be calculated.

    Returns :
        chi2: *float*
            Chi2 value for given model parameters.
    """
    # First we have to change the values of parameters in
    # event.model.parameters to values given by theta.
    for (parameter, value) in zip(parameters_to_fit, theta):
        setattr(event.model.parameters, parameter, value)
    event.fit_fluxes()  # Fit source and blend fluxes to the data
    # After that, calculating chi2 is trivial:
    return event.get_chi2()

def ln_like(theta, event, parameters_to_fit):
    """
    Calculate the log-likelihood. For a Gaussian log-likelihood function, this
    is equal to -0.5 * chi2.
    """
    for (parameter, value) in zip(parameters_to_fit, theta):
        setattr(event.model.parameters, parameter, value)
    chi2 = event.get_chi2()
    if chi2 < ln_like.best[0]:
        # Functions can have attributes. We have an attribute called "best",
        # which contains the smallest chi2 and the corresponding parameters.
        ln_like.best = [chi2, theta]
    return -0.5 * chi2

# Create the "best" attribute.
ln_like.best = [np.inf]

def ln_prior(theta, parameters_to_fit, event):
    """
    Calculate the log-prior.
    t_E and rho are not allowed to be negative,
    and we have a condition that disfavors negative blending (fs > 1).
    """
    if theta[parameters_to_fit.index('t_E')] < 0.: return -np.inf
    d = dict(zip(parameters_to_fit, theta))
    if 'rho' in d and d['rho'] < 1e-6: return -np.inf

    for (parameter, value) in zip(parameters_to_fit, theta):
        setattr(event.model.parameters, parameter, value)
    event.fit_fluxes()
    source_flux = event.source_fluxes[0]
    blend_flux = event.blend_fluxes[0]
    fs = source_flux/(source_flux + blend_flux)

    if fs > 1:
         mean = 1.
         sigma = 0.1
         return -0.5 * ((fs - mean) / sigma)**2
    return 0.0

def ln_prob(theta, event, parameters_to_fit):
    """
    Calculate the log-posterior. Since the product of the prior and the likelihood
    is proportional to the posterior, the sum of the log-prior and the
    log-likelihood is proportional to the log-posterior
    """
    ln_prior_ = ln_prior(theta, parameters_to_fit, event)
    event.fit_fluxes()
    source_flux = event.source_fluxes[0]
    blend_flux = event.blend_fluxes[0]
    fs = source_flux/(source_flux + blend_flux)

    if not np.isfinite(ln_prior_): return (-np.inf, fs)
    ln_like_ = ln_like(theta, event, parameters_to_fit)
    if np.isnan(ln_like_): return (-np.inf, fs)

    return (ln_prior_ + ln_like_, fs)

#------------ Data binning functions to speed up higher order effect exploration -----------------
def bin_lightcurve(data, bin_width_days):
    t, m, e = data['time'].values, data['magF146'].values, data['magF146err'].values
    bins = np.arange(t.min(), t.max() + bin_width_days, bin_width_days)
    bin_idx = np.digitize(t, bins)
    t_bin, m_bin, e_bin = [], [], []
    for i in range(1, len(bins)):
        mask = bin_idx == i
        if mask.sum() == 0: continue
        w = 1.0 / e[mask]**2
        t_bin.append(np.average(t[mask]))
        m_bin.append(np.average(m[mask], weights=w))
        e_bin.append(1.0 / np.sqrt(w.sum()))
    return pd.DataFrame({'time': t_bin, 'magF146': m_bin, 'magF146err': e_bin})

def reduce_data_to_speedup(data, bin_width_days, peak=None, t_range=None):
    if t_range is not None:
        t_min, t_max = t_range
        if t_min is not None: data = data[data['time'] >= t_min]
        if t_max is not None: data = data[data['time'] <= t_max]
        data = data.reset_index(drop=True)

    if peak is not None:
        t_peak, t_fwhm, peak_bin_width = peak
        near = (data['time'] >= t_peak - t_fwhm) & (data['time'] <= t_peak + t_fwhm)
        data_peak, data_outer = data[near], data[~near]
        if peak_bin_width and len(data_peak) > 0:
            data_peak = bin_lightcurve(data_peak, peak_bin_width)
    else:
        data_peak = pd.DataFrame(columns=data.columns)
        data_outer = data

    binned_outer = bin_lightcurve(data_outer, bin_width_days)
    return pd.concat([binned_outer, data_peak]).sort_values('time').reset_index(drop=True)

# Available finite source magnification methods for FSPL models
# Method index 1: finite_source_uniform_Gould94_direct  (1e-3 < rho < 1, recommended)
finite_source_methods = [
    'finite_source_uniform_Gould94',         # 0: 1e-3 < rho < 1 (has a bug)
    'finite_source_uniform_Gould94_direct',  # 1: 1e-3 < rho < 1  <-- recommended
    'finite_source_uniform_WittMao94',       # 2: rho < 0.01
    'finite_source_uniform_Lee09',           # 3: rho > 0.01
    'finite_source_LD_WittMao94',            # 4: rho < 0.01 (limb-darkened)
    'finite_source_LD_Yoo04',               # 5: 1e-3 < rho < 1  (limb-darkened)
    'finite_source_LD_Yoo04_direct',        # 6: 1e-3 < rho < 1  (limb-darkened)
    'finite_source_LD_Lee09',               # 7: rho > 0.01       (limb-darkened)
]

## Section 1 — Load and Visualize the Data

In [ ]:
# TODO: Load the data using pandas
# Hint: Use pd.read_csv()
path = pointlensdata_path
filename = 'GrpProjData<your_number>.csv'  # ← change to your assigned file
data = ___
display(data.head())

# TODO: Load into MulensModel
# REMINDER: mm.MulensData handles formatting of the raw arrays
my_data = mm.MulensData(
    data_list=[data['time'].values, ___, ___],
    phot_fmt='mag'
)

# TODO: Plot the light curve using my_data.plot() and


## Section 2 — Rough estimate of initial parameters

In [ ]:
# TODO: Estimate initial parameters t_0, u_0, t_E
# Hint: t_0 is approximately the time corresponding to the minimum magnitude
t_0_guess = ___

# Hint: u_0 can typically be between 0 and 1
u_0_guess = ___

# Hint: t_E is roughly the duration of the event (e.g. 20-30 days)
t_E_guess = ___

print(f"Estimated t_0: {t_0_guess:.2f}, t_E: {t_E_guess:.2f}, u_0: {u_0_guess:.2f}")

## Key idea in light curve fitting: Minimization vs. MCMC
Microlensing light curve fitting involves evaluating the likelihood space, which can be highly degenerate and complex. To solve these models efficiently, we can follow a strict 2-step workflow:

1. **Minimizer First:** We use an optimizer (like `iminuit` or `scipy.optimize`) first to rapidly descend into the global minimum of our $\chi^2$ surface. This gives us the absolute "best-fit" approximate parameters, but minimizers are poor at estimating the *errors* (uncertainties) of these parameters.
2. **MCMC Second:** Markov Chain Monte Carlo (MCMC) is a *sampler*, not an optimizer. Its job is to map out the posterior distribution (error bars and correlations). If you start MCMC walkers randomly, they will spend immense amounts of computational time wandering around looking for the minimum. By initializing walkers in a tight ball exactly *around* the minimizer's best-fit result, MCMC can immediately start exploring the local parameter space, yielding accurate uncertainties and beautiful corner plots.

## Section 3 — Fit a PSPL model (Minimizer)

In [ ]:
# TODO: Set up the initial model and event (for plotting the guess later)
# Hint: Use mm.Model and mm.Event as before. Call fit_fluxes() to align the baseline.
init_pspl_model = mm.Model({'t_0': ___, 'u_0': ___, 't_E': ___})
init_event_pspl = mm.Event(datasets=___, model=___)
init_event_pspl.___()  # Fit source and blend fluxes to the data

# TODO: Set up the model and event that we will actually minimize
pspl_model = mm.Model({'t_0': ___, 'u_0': ___, 't_E': ___})
event_pspl = mm.Event(datasets=___, model=___)
event_pspl.___()  # Fit source and blend fluxes to the data

pspl_params = ['t_0', 'u_0', 't_E']

# TODO: Define bounds for the minimizer
# Hint: t_0 and u_0 can be unbounded (None, None). t_E should be strictly positive (e.g., 1e-6 to None). Thus eg. bounds_pspl = [(None, None), (None, None), (1e-6, None)]
bounds_pspl = [___, ___, ___]

# TODO: Minimize chi2 to find best-fit parameters using iminuit
# Hint: Pass chi2_fun, initial guesses (x0), args=(pspl_params, event_pspl), and bounds
m_pspl = minimize(___, x0=[___, ___, ___], args=___, bounds=___)
best_pspl = m_pspl.x

# TODO: Extract the source flux fraction (fs) for the first dataset
# Hint: fs = source_flux / (source_flux + blend_flux)
fs = ___

# TODO: Calculate Number of Degrees of Freedom (dof) and the minimum chi2
# Hint: dof = number of data points (n_epochs) - number of fitted parameters
dof_pspl = ___
chi2min_pspl = chi2_fun(___, ___, ___)

# TODO: Calculate the z-score
# Hint: For Ndof >> 50, the chi2 distribution is nearly normal. z = (chi2 - dof) / sqrt(2 * dof)
z_score_pspl = ___

In [ ]:
# TODO: Plot the initial model, best-fit model, and data
# Hint: Ensure subtract_2450000=True is passed to all plot functions to align the x-axis
event_pspl.plot_model(t_range=(data['time'].min(), data['time'].max()), color='red', subtract_2450000=True, zorder=4, label='Best-fit PSPL')
init_event_pspl.plot_model(t_range=___, color='blue', subtract_2450000=___, zorder=3, label='Initial PSPL')
init_event_pspl.plot_data(subtract_2450000=___, color='gray', marker='.', linestyle='None', markersize=2)

# TODO: Fill in the variables for the text box display
plt.text(0.05, 0.95, f'Best-fit PSPL Parameters:\n' +
         '\n'.join([f'{param}: {val:.4f}' for param, val in zip(pspl_params, best_pspl)]) +
         f'\n\nfs: {___:.4f}' +
         f'\n\n$\chi^2_{{min}}$: {___:.2f}\nDOF: {___}\n$\chi^2_{{min}}$/DOF: {___/___:.3f}'+
         f'\n\nz-score: {___:.2f}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.legend()
plt.xlabel('JD - 2450000', fontsize=12)
plt.ylabel(r'$m_{F_{146}}$ [mag]', fontsize=12)
plt.title('Best-fit PSPL Model vs Data', fontsize=14)
plt.show()

## Section 4 — MCMC run for PSPL

In [ ]:
n_dim = len(pspl_params)

# TODO: Set MCMC hyperparameters
# Hint: recommended n_walkers is 2-4 times the number of dimensions (n_dim)
n_walkers = ___
# Hint: recommended n_steps is 1000-3000
n_steps = ___
# Hint: recommended n_burn (burn-in steps to discard) is 100-500
n_burn = ___
# Hint: can be kept at 1 (thinning factor to reduce autocorrelation)
thin = ___

# TODO: Initialize walkers with a small perturbation around the minimizer's best-fit parameters
# Hint: Use best_pspl and add a small random scatter, say 1e-4, using np.random.randn(n_walkers, n_dim)
pos = ___ + 1e-4 * np.random.randn(___, ___)

# TODO: Initialize the MCMC sampler
# Hint: Use emcee.EnsembleSampler. Pass n_walkers, n_dim, the ln_prob function, and args=(event_pspl, pspl_params)
sampler_pspl = emcee.EnsembleSampler(___, ___, ___, args=(___, ___))

# TODO: Run the MCMC sampler: this step will show the progress bar in the notebook and it may take a few minutes to complete
# Hint: Use run_mcmc and pass the initial positions (pos) and the number of steps (n_steps)
sampler_pspl.run_mcmc(___, ___, progress=True)

In [ ]:
# Extract the chains and blobs (blobs contain our fs parameter calculated in ln_prob)
# TODO: Pass your n_burn and thin variables to the discard and thin arguments below
samples_pspl = np.concatenate([sampler_pspl.get_chain(), sampler_pspl.get_blobs()[:, :, np.newaxis]], axis=2)
flat_samples_pspl = np.concatenate([sampler_pspl.get_chain(discard=n_burn, thin=thin, flat=True),
                                   sampler_pspl.get_blobs(discard=n_burn, thin=thin, flat=True)[:, np.newaxis]], axis=1)

# TODO: Calculate the median and standard deviation (errors) for the parameters
med_pspl = np.median(flat_samples_pspl, axis=0)
err_pspl = np.std(flat_samples_pspl, axis=0)

In [ ]:
# TODO: Define the list of parameters including the source flux fraction 'fs'
pspl_params_with_fs = pspl_params + ['fs']
nrows = len(pspl_params_with_fs)

# TODO: Plot the MCMC chains to check for convergence
fig, axes = plt.subplots(nrows, 1, figsize=(10, 1*nrows), sharex=True)
for i in range(nrows):
    axes[i].plot(samples_pspl[:, :, i], 'k', alpha=0.3)
    axes[i].axhline(med_pspl[i], color='green', linestyle='-')
    axes[i].axvline(n_burn, color='blue', linestyle='--')
    axes[i].set_ylabel(pspl_params_with_fs[i])
axes[-1].set_xlabel('step number')
fig.suptitle(f'PSPL MCMC Chains', fontsize=14)
plt.tight_layout()
plt.show()

# TODO: Plot the corner plot of the posterior distributions
# Hint: use corner.corner() with flat_samples_pspl, and set truths to the medians
fig = corner.corner(flat_samples_pspl, labels=pspl_params_with_fs, truths=med_pspl,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True, title_fmt='.4f')
plt.show()

# TODO: Calculate chi2, DOF, and z-score for the MCMC median parameters
chi2_mcmc_pspl = ___
dof_pspl = ___
z_score_mcmc_pspl = ___

# Plot the data and the best-fit model from MCMC
plt.figure()
# TODO: Plot data and model
# Hint: Remember to use subtract_2450000=True
event_pspl.plot_data(___)
event_pspl.plot_model(___)

# TODO: Add text box with best-fit parameters, errors, and chi2 information
# Hint: Ensure your variables map correctly into the text string (params, medians, errors)
plt.text(1.05, 0.95, f'MCMC Best-fit PSPL Parameters:\n' +
         '\n'.join([f'{param}: {val:.4f} ± {e:.4f}' for param, val, e in zip(___, ___, ___)]) +
         f'\n\n$\chi^2$: {___:.2f}\nDOF: {___}\n$\chi^2$/DOF: {___/___:.3f}'+
         f'\n\nz-score: {___:.2f}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.legend(loc='upper left')
plt.xlabel('JD - 2450000', fontsize=12)
plt.ylabel(r'$m_{F_{146}}$ [mag]', fontsize=12)
plt.title(f'PSPL Final MCMC Best-Fit: {filename}', fontsize=14)
plt.show()

## Now, data binning is needed to explore Higher order effects faster
When fitting higher-order effects like Finite Source (FSPL) or Parallax, generating the model requires computationally intensive integration (e.g., over the stellar disk or the Earth's orbit).

To ensure the minimizer and MCMC converge quickly, we apply a **strategic data reduction/binning technique** via the `reduce_data_to_speedup` helper function:
1. **Inverse-Variance Weighting:** The `bin_lightcurve` function groups observations within a specific time window. It calculates the mean time, and computes a weighted average of the magnitudes using $1/\sigma^2$, yielding a single robust data point with properly propagated errors.
2. **Preserving the Peak for FS effect:** Finite source effects generally manifest strongly only near $t_0$. Therefore, regions far from $t_0$ are aggressively binned (e.g., 1-day bins) to save time, while the "peak" region is either left unbinned or lightly binned to preserve the shape necessary to accurately fit $\rho$.
3. **Keeping the peak season:** We can keep the peak season and chop off the other season while FS effect is explored. However, while exploring parallax Keep all the season for maximizing the parallax signal.

## Section 5 — Exploring Higher Order Effects

Choose **one** of the three models below by uncommenting it.
Make sure data is binned before running FSPL or parallax fits.

| Choice | Extra params | Difficulty-level |
|--------|-------------|------------------|
| **FSPL** | $\rho$ | Low |
| **PSPL+Parallax** | $\pi_{E,E}$, $\pi_{E,N}$ | Low |
| **FSPL+Parallax** | $\rho$, $\pi_{E,E}$, $\pi_{E,N}$ | High |

**TODO:** Uncomment your chosen model.

In [ ]:
# TODO: Extract your best-fit PSPL parameters to use as initial guesses
# Hint: Unpack best_pspl
t_0_guess, u_0_guess, t_E_guess = ___

# TODO: Bin the data and limit the time range to speed up higher-order effect exploration
binned_df = reduce_data_to_speedup(data, bin_width_days=1, t_range=(t_0_guess - 5*t_E_guess, t_0_guess + 5*t_E_guess), peak=(t_0_guess, t_E_guess, 0.1))

# TODO: Wrap the binned data into MulensData
binned_data = mm.MulensData(
    data_list=[binned_df['time'].values, ___, ___],
    phot_fmt='mag'
)

# Visualize the binned data
plt.figure()
binned_data.plot(___)
plt.legend()
plt.ylabel(r'$m_{F_{146}}$ [mag]', fontsize=12)
plt.xlabel('JD - 2450000', fontsize=12)
plt.title('Binned Data for Higher Order Effect Exploration', fontsize=14)
plt.show()

In [ ]:
# ── TODO: uncomment ONE of the three options below ───────────────────────────

# Option A — FSPL
# rho_guess = 0.5
# FS_METHOD = finite_source_methods[1]
# model_ho   = mm.Model({'t_0': t_0_guess, 'u_0': u_0_guess, 't_E': t_E_guess, 'rho': rho_guess})
# model_ho.set_magnification_methods([binned_df['time'].min(), FS_METHOD,
#                                      binned_df['time'].max()])
# model_label = 'FSPL'
# labels_ho = ['t_0', 'u_0', 't_E', 'rho']

# Option B — PSPL + Parallax
# pi_E_E_guess, pi_E_N_guess = 0.1, 0.1
# ra, dec = "17:52:41.16", "-29:16:46.29"
# t_0_ref = ___# will be provided with the data
# model_ho = mm.Model({'t_0': t_0_guess, 'u_0': u_0_guess, 't_E': t_E_guess,
#                      'pi_E_E': pi_E_E_guess, 'pi_E_N': pi_E_N_guess, 't_0_ref': t_0_ref}, coords=ra+' '+dec)
# model_ho.parallax(earth_orbital=True)
# model_label = 'PSPL+Parallax'
# labels_ho = ['t_0', 'u_0', 't_E', 'pi_E_E', 'pi_E_N']

# Option C — FSPL + Parallax
# rho_guess = 0.5
# FS_METHOD = finite_source_methods[1]
# pi_E_E_guess, pi_E_N_guess = 0.1, 0.1
# ra, dec = "17:52:41.16", "-29:16:46.29"
# t_0_ref = ___# will be provided with the data
# model_ho = mm.Model({'t_0': t_0_guess, 'u_0': u_0_guess, 't_E': t_E_guess, 'rho': rho_guess,
#                      'pi_E_E': pi_E_E_guess, 'pi_E_N': pi_E_N_guess, 't_0_ref': t_0_ref}, coords=ra+' '+dec)
# model_ho.set_magnification_methods([binned_df['time'].min(), FS_METHOD,
#                                      binned_df['time'].max()])
# model_ho.parallax(earth_orbital=True)
# model_label = 'FSPL+Parallax'
# labels_ho = ['t_0', 'u_0', 't_E', 'rho', 'pi_E_E', 'pi_E_N']

# ─────────────────────────────────────────────────────────────────────────────

# TODO: Set up the higher order event and fit baseline fluxes
event_ho = mm.Event(datasets=___, model=___)
event_ho.___()  # Fit source and blend fluxes to the data

# TODO: Keep a copy of the initial model for comparison plotting later
# Hint: Set up init_ho_model identically to your chosen model_ho above
init_ho_model = mm.Model({'t_0': ___, 'u_0': ___, 't_E': ___, 'rho': ___}) # Adjust dictionary if using parallax
init_ho_model.set_magnification_methods([binned_df['time'].min(), ___, binned_df['time'].max()])
init_event_ho = mm.Event(datasets=___, model=___)
init_event_ho.___()

# TODO: Define the list of parameters to fit and their minimizer bounds
# Hint: Use (None, None) for unbounded, and (1e-6, None) for parameters that must be strictly positive (t_E, rho)
ho_params = ___
bounds_ho = [___, ___, ___, ___]  # Add more bounds if fitting parallax!

# TODO: Minimize chi2 to find best-fit parameters
# Hint: Use minimize with chi2_fun, your initial guesses (x0), args=(ho_params, event_ho), and bounds
m_ho = minimize(___, x0=[___, ___, ___, ___], args=___, bounds=___)
best_ho = m_ho.x

# TODO: Extract fs, calculate DOF, min chi2, and z-score for the higher order model
fs = (event_ho.___ / (event_ho.___ + event_ho.___))[0]
dof_ho = event_ho.datasets[0].___ - len(___)
chi2min_ho = ___
z_score_ho = ___

# Plot initial model, best-fit model, and data
event_ho.plot_model(___)
init_event_ho.plot_model(___)
init_event_ho.plot_data(___)

# TODO: Fill in the text box variables
plt.text(0.05, 0.95, f'Best-fit {model_label} Parameters:\n' +
         '\n'.join([f'{param}: {val:.4f}' for param, val in zip(___, ___)]) +
         f'\n\nfs: {___:.4f}' +
         f'\n\n$\chi^2_{{min}}$: {___:.2f}\nDOF: {___}\n$\chi^2_{{min}}$/DOF: {___/___:.3f}'+
         f'\n\nz-score: {___:.2f}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.legend()
plt.xlabel('JD - 2450000', fontsize=12)
plt.ylabel(r'$m_{F_{146}}$ [mag]', fontsize=12)
plt.title(f'Best-fit {model_label} Model vs Data', fontsize=14)
plt.show()

## Section 6 — Compare Models

In [ ]:
# TODO: Check if the higher-order effect model is preferred over the PSPL model based on z-score comparison
if z_score_ho < z_score_pspl:
    print(f"Higher-order effect model is preferred over PSPL model based on z-score comparison: {z_score_ho:.2f} < {z_score_pspl:.2f}")

## Section 7 — OPTIONAL: Run MCMC for Higher Order Model

In [ ]:
# TODO: Set MCMC hyperparameters based on the number of dimensions
n_dim = len(ho_params)
n_walkers = 4 * n_dim # recommended number of walkers is 2-4 times the number of dimensions
n_steps = 2000 # recommended number of steps is 1000-3000
n_burn = 200 # recommended number of burn-in steps is 100-500
thin = 1 #

# TODO: Initialize walkers in a small ball (perturbation) around the minimizer's best-fit parameters
# Hint: Use best_ho and add a small random scatter using np.random.randn(n_walkers, n_dim)
pos = best_ho + 1e-4 * np.random.randn(n_walkers, n_dim)

# TODO: Initialize the MCMC sampler
# Hint: Use emcee.EnsembleSampler. Pass n_walkers, n_dim, ln_prob, and args=(event_ho, ho_params)
sampler_ho = emcee.EnsembleSampler(___, ___, ___, args=(___, ___))

# TODO: Run the MCMC sampler
# Hint: Pass the initial positions (pos) and the number of steps (n_steps)
sampler_ho.run_mcmc(___, ___, progress=True)

# Extract the chains and blobs (blobs contain our fs parameter calculated in ln_prob)
# TODO: Pass your n_burn and thin variables to the discard and thin arguments below
samples_ho = np.concatenate([sampler_ho.get_chain(), sampler_ho.get_blobs()[:, :, np.newaxis]], axis=2)
flat_samples_ho = np.concatenate([
    sampler_ho.get_chain(discard=___, thin=___, flat=True),
    sampler_ho.get_blobs(discard=___, thin=___, flat=True)[:, np.newaxis]
], axis=1)

# TODO: Calculate the median and standard deviation (errors) for the parameters
# Hint: Use np.median and np.std along axis=0 on the flat_samples_ho array
med_ho = np.median(___, axis=___)
err_ho = np.std(___, axis=___)

In [ ]:
# TODO: Define the list of parameters including the source flux fraction 'fs'
ho_params_with_fs = ho_params + ['fs']
nrows = len(ho_params_with_fs)

# TODO: Plot the MCMC chains to check for convergence
# Hint: Create subplots and loop over the parameters
fig, axes = plt.subplots(nrows, 1, figsize=(10, 1*nrows), sharex=True)
for i in range(nrows):
    # Hint: use the unflattened samples array (samples_ho)
    axes[i].plot(samples_ho[:, :, i], 'k', alpha=0.3)
    # Hint: plot the median value as a horizontal line
    axes[i].axhline(med_ho[i], color='green', linestyle='-')
    # Hint: plot the burn-in cutoff as a vertical line (n_burn)
    axes[i].axvline(n_burn, color='blue', linestyle='--')
    axes[i].set_ylabel(ho_params_with_fs[i])
axes[-1].set_xlabel('step number')
fig.suptitle(f'HO MCMC Chains', fontsize=14)
plt.tight_layout()
plt.show()

# TODO: Plot the corner plot of the posterior distributions
# Hint: use corner.corner() with flat_samples_ho, and set truths to the medians
fig = corner.corner(___)
plt.show()

# TODO: Calculate chi2, DOF, and z-score for the MCMC median parameters
chi2_mcmc_ho = ___
dof_ho = ___
z_score_mcmc_ho = ___

# Plot the data and the best-fit model from MCMC
plt.figure()

# TODO: Plot data and model
# Hint: Remember to use subtract_2450000=True
event_ho.plot_data(___)
event_ho.plot_model(___)

# TODO: Add text box with best-fit parameters, errors, and chi2 information
# Hint: Ensure your variables map correctly into the text string (params, medians, errors)
plt.text(1.05, 0.95, f'MCMC Best-fit {model_label} Parameters:\n' +
         '\n'.join([f'{param}: {val:.4f} ± {e:.4f}' for param, val, e in zip(___, ___, ___)]) +
         f'\n\n$\chi^2$: {___:.2f}\nDOF: {___}\n$\chi^2$/DOF: {___/___:.3f}'+
         f'\n\nz-score: {___:.2f}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.legend(loc='upper left')
plt.xlabel('JD - 2450000', fontsize=12)
plt.ylabel(r'$m_{F_{146}}$ [mag]', fontsize=12)
plt.title(f'{model_label} Final MCMC Best-Fit: {filename}', fontsize=14)
plt.show()

---
# ======================= END =======